# 00 · Datos, presentaciones y equivalencias

**Objetivo:** Medir con precios reales cuánto sobreprecio sostiene la marca líder frente a su competidor directo, como caso de valor de marca para directores de marketing y ventas.

**Pregunta:** En la misma tienda, ¿cuánto más cuesta Coca-Cola que Pepsi en presentación equivalente?

Este notebook responde: ¿qué presentaciones de Coca-Cola y Pepsi hay, en cuántas tiendas, cuáles son
equivalentes y cómo se comparan? Cierra con la tabla de decisiones metodológicas (plan aprobado en
`_specs/plan_metodologico_6_12.md`).

**Prima** = cuánto más cuesta Coca-Cola que Pepsi en la misma tienda, en % y en pesos. Se habla de precio relativo
sostenido, no de «valor de marca» en sentido contable.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, pares, estadistica as E, charts as C
import prima as P
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
reg = entorno.registrar
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
pts, st, cat = P.load()
REPS = 500
raw = entorno.load_raw()
manifest = entorno.write_manifest(raw)
x = cat[cat.familia.isin(["Coca Cola", "Pepsi"])].copy()
cov = pts.merge(st[["establecimiento_id", "giro"]], on="establecimiento_id")
cov = cov[cov.sku_id.isin(x.sku_id)].groupby(["sku_id", "giro"]).establecimiento_id.nunique().unstack(fill_value=0)
tab = x.set_index("sku_id")[["marca", "presentacion", "cantidad_base", "empresa_propietaria"]].join(cov).fillna(0)
display(tab.sort_values(["marca", "cantidad_base"]))

,marca,presentacion,cantidad_base,empresa_propietaria,Central de Abasto,Farmacias,Mercados,Supermercado / Tienda de Autoservicio,Tienda de Conveniencia
sku_id,,,,,,,,,
2588,Coca Cola,Lata 355 Ml.,0.355,The Coca-Cola Company,11,12,19,255,42
338,Coca Cola,Botella 600 Ml.,0.600,The Coca-Cola Company,0,15,0,280,47
337,Coca Cola,Botella 1 Lt. No Retornable,1.000,The Coca-Cola Company,2,12,3,202,25
3593,Coca Cola,Botella 1.75 Lt. No Retornable,1.750,The Coca-Cola Company,0,0,0,8,3
3210,Coca Cola,Botella 2.75 Lt. No Retornable,2.750,The Coca-Cola Company,0,5,0,162,12
339,Coca Cola. Light,Botella 600 Ml.,0.600,The Coca-Cola Company,0,15,0,274,42
2498,Coca Cola. Light,Botella 1 Lt. No Retornable,1.000,The Coca-Cola Company,0,7,0,116,12
2589,Coca Cola. Light,Botella 2 Lt. No Retornable,2.000,The Coca-Cola Company,0,6,0,195,11
340,Coca Cola. Zero,Botella 600 Ml.,0.600,The Coca-Cola Company,0,16,0,267,40


## 1. ¿Qué presentaciones equivalen?

La ficha citaba Coca-Cola 600 ml en 342 tiendas y Pepsi en 306. En el archivo **no hay Pepsi de 600 ml**: Pepsi
regular solo viene en lata de 355 ml, botella de 2.5 L y de 3 L.

**Decisiones:**
1. **Comparación principal: lata 355 ml contra lata 355 ml.** Mismo tamaño y mismo envase; el precio por empaque es
   también el precio por litro. Es el par limpio.
2. **Botella familiar: Coca-Cola 2.75 L contra Pepsi 3 L, por litro.** Son las presentaciones grandes que más tiendas
   tienen (162 y 254 en supermercados). Por litro, porque los tamaños difieren (razón 0.92); el precio por litro favorece
   un poco al envase más grande, así que también se muestra por empaque.
3. **Sensibilidad: Coca-Cola Light 2 L contra Pepsi Light 2.5 L,** por litro (pocas tiendas con las dos).

**Alternativas descartadas:**
- Coca-Cola 600 ml contra cualquier Pepsi: no hay presentación equivalente.
- Coca-Cola 2.75 L contra Pepsi 2.5 L: Pepsi 2.5 L está en solo 10 supermercados.
- Promedio nacional de cada marca: mezcla tiendas y ciudades.

**Misma tienda** (`pares.marcas`): controla ciudad, cadena y día de levantamiento; es lo que el cliente ve frente al
refrigerador. Precio de tienda = mediana de la quincena.

In [2]:
print("Empresa propietaria verificada (sku_catalogo):")
display(cat[cat.familia.isin(["Coca Cola", "Pepsi"])].drop_duplicates("familia")[["familia", "empresa_propietaria",
        "fuente_propietaria", "url_fuente_propietaria"]])

Empresa propietaria verificada (sku_catalogo):


,familia,empresa_propietaria,fuente_propietaria,url_fuente_propietaria
337,Coca Cola,The Coca-Cola Company,"Form 10-K 2025, Item 1: «We own and market numerous valuable bever...",https://www.sec.gov/Archives/edgar/data/21344/000162828026010047/k...
344,Pepsi,"PepsiCo, Inc.",Form 10-K 2025: «We own numerous valuable trademarks… including… P...,https://www.sec.gov/Archives/edgar/data/77476/000007747626000007/p...


## 2. Canal

**Decisión:** supermercados y autoservicio, donde están casi todos los pares. Farmacias y tiendas de conveniencia se
muestran como contexto: tienen muy pocas tiendas con las dos marcas.

In [3]:
cob = []
for k, c in P.COMPARA.items():
    for canal in [P.CANAL, "Farmacias", "Tienda de Conveniencia"]:
        p = P.pairs(pts, st, cat, k, canal=canal)
        cob.append({"comparación": c["nombre"], "canal": canal, "tiendas con las dos": len(p),
                    "municipios": p.municipio_id.nunique() if len(p) else 0})
cob = pd.DataFrame(cob)
display(cob)

,comparación,canal,tiendas con las dos,municipios
0,Lata 355 ml vs. lata 355 ml,Supermercado / Tienda de Autoservicio,184,56
1,Lata 355 ml vs. lata 355 ml,Farmacias,9,9
2,Lata 355 ml vs. lata 355 ml,Tienda de Conveniencia,10,8
3,Coca-Cola 2.75 L vs. Pepsi 3 L,Supermercado / Tienda de Autoservicio,144,40
4,Coca-Cola 2.75 L vs. Pepsi 3 L,Farmacias,5,5
5,Coca-Cola 2.75 L vs. Pepsi 3 L,Tienda de Conveniencia,7,6
6,Coca-Cola Light 2 L vs. Pepsi Light 2.5 L,Supermercado / Tienda de Autoservicio,36,18
7,Coca-Cola Light 2 L vs. Pepsi Light 2.5 L,Farmacias,1,1
8,Coca-Cola Light 2 L vs. Pepsi Light 2.5 L,Tienda de Conveniencia,0,0


## 3. Calidad

- Posibles errores y extremos (`estadistica.flag_prices` sobre el precio de tienda): se marcan y la prima principal se
  repite sin ellos.
- Promociones de un día: el precio de tienda es la mediana de la quincena, que las atenúa; sensibilidad solo con tiendas
  donde el precio no cambió en la quincena.

In [4]:
sm = pts.merge(st[st.giro == P.CANAL][["establecimiento_id"]], on="establecimiento_id")
f = E.flag_prices(sm, sku="sku_id")
ids = [c[s] for c in P.COMPARA.values() for s in ("coca", "pepsi")]
q = f[f.sku_id.isin(ids)].groupby("sku_id")[["posible_error", "extremo"]].mean().mul(100)
q["cambia_en_quincena_%"] = sm[sm.sku_id.isin(ids)].groupby("sku_id").precios_distintos.apply(lambda s: (s > 1).mean() * 100)
display(q.join(cat.set_index("sku_id")[["marca", "presentacion"]]).round(1))

,posible_error,extremo,cambia_en_quincena_%,marca,presentacion
sku_id,,,,,
344,0.0,0.0,16.1,Pepsi,Botella 3 Lt. No Retornable
1149,0.0,0.0,4.5,Pepsi,Lata 355 Ml.
1150,0.0,0.0,14.9,Pepsi. Light,Botella 2.5 Lt. No Retornable
2588,0.0,0.0,14.9,Coca Cola,Lata 355 Ml.
2589,0.0,0.0,10.8,Coca Cola. Light,Botella 2 Lt. No Retornable
3210,0.0,0.0,18.5,Coca Cola,Botella 2.75 Lt. No Retornable


In [5]:
cob.to_csv(PROC / "cobertura_comparaciones.csv", index=False, encoding="utf-8-sig")
params = {"semilla": P.SEMILLA, "bootstrap_replicas": REPS, "canal": P.CANAL, "min_pares": P.MIN_PARES,
          "comparaciones": P.COMPARA}
_ = (META / "parametros.json").write_text(json.dumps(params, ensure_ascii=False, indent=2), encoding="utf-8")
F = "00_datos_y_cobertura"
for k in P.COMPARA:
    r_ = cob[(cob.comparación == P.COMPARA[k]["nombre"]) & (cob.canal == P.CANAL)].iloc[0]
    reg(f"cob_{k}", r_["tiendas con las dos"], fuente=F); reg(f"cob_{k}_mun", r_.municipios, fuente=F)
reg("tiendas_coca600", tab.loc[338, P.CANAL], fuente=F)
reg("tiendas_coca_lata", tab.loc[2588, P.CANAL], fuente=F); reg("tiendas_pepsi_lata", tab.loc[1149, P.CANAL], fuente=F)

{'cob_lata': {'cifra': 184.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_cobertura',
  'nota': ''},
 'cob_lata_mun': {'cifra': 56.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_cobertura',
  'nota': ''},
 'cob_familiar': {'cifra': 144.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_cobertura',
  'nota': ''},
 'cob_familiar_mun': {'cifra': 40.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_cobertura',
  'nota': ''},
 'cob_light': {'cifra': 36.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_cobertura',
  'nota': ''},
 'cob_light_mun': {'cifra': 18.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_cobertura',
  'nota': ''},
 'tiendas_coca600': {'cifra': 280.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_cobertura',
  'nota': ''},
 'tiendas_coca_lata': {'cifra': 255.0,
  'intervalo': None,
  'n': None,
  'fuente': '00_datos_y_cobertura',
  'nota': ''},
 'tiendas_pepsi_lata': {'cifra': 201.0,
  'int

## 4. Decisiones metodológicas

In [6]:
display(Markdown(f'''
| # | Decisión | Propuesta | Por qué | Alternativas descartadas |
|---|---|---|---|---|
| 1 | Par principal | lata 355 ml vs. lata 355 ml, precio por empaque | mismo tamaño y envase: el par limpio | Coca-Cola 600 ml (no hay Pepsi de 600 ml) |
| 2 | Botella familiar | Coca-Cola 2.75 L vs. Pepsi 3 L por litro; también por empaque | son las grandes con más tiendas; tamaño distinto (0.92) | Pepsi 2.5 L (10 tiendas) |
| 3 | Sensibilidad | Coca-Cola Light 2 L vs. Pepsi Light 2.5 L por litro | variante sin azúcar | — |
| 4 | Comparación | misma tienda (`pares.marcas`); precio de tienda = mediana de la quincena | controla ciudad, cadena y día | promedio nacional de cada marca |
| 5 | Prima | mediana de log(Coca-Cola / Pepsi) en % y mediana de la diferencia en pesos; % invertidos | robusta a extremos; casi no hay empates | promedio simple |
| 6 | Incertidumbre | IC90 por bootstrap de dos vías (tiendas y municipios), 500 réplicas, semilla 2026 | las tiendas de una ciudad no son independientes | IC por par |
| 7 | Desglose | cadena, formato, grupo y región con 10 pares o más | ¿dónde se reduce o se invierte? | — |
| 8 | Calidad | extremos y promociones marcados; prima repetida sin ellos | no borrar en silencio | borrar extremos |
| — | Alcance | precio relativo sostenido, no valor contable de la marca | solo hay precios al consumidor | inferir margen o ventas |
'''))


| # | Decisión | Propuesta | Por qué | Alternativas descartadas |
|---|---|---|---|---|
| 1 | Par principal | lata 355 ml vs. lata 355 ml, precio por empaque | mismo tamaño y envase: el par limpio | Coca-Cola 600 ml (no hay Pepsi de 600 ml) |
| 2 | Botella familiar | Coca-Cola 2.75 L vs. Pepsi 3 L por litro; también por empaque | son las grandes con más tiendas; tamaño distinto (0.92) | Pepsi 2.5 L (10 tiendas) |
| 3 | Sensibilidad | Coca-Cola Light 2 L vs. Pepsi Light 2.5 L por litro | variante sin azúcar | — |
| 4 | Comparación | misma tienda (`pares.marcas`); precio de tienda = mediana de la quincena | controla ciudad, cadena y día | promedio nacional de cada marca |
| 5 | Prima | mediana de log(Coca-Cola / Pepsi) en % y mediana de la diferencia en pesos; % invertidos | robusta a extremos; casi no hay empates | promedio simple |
| 6 | Incertidumbre | IC90 por bootstrap de dos vías (tiendas y municipios), 500 réplicas, semilla 2026 | las tiendas de una ciudad no son independientes | IC por par |
| 7 | Desglose | cadena, formato, grupo y región con 10 pares o más | ¿dónde se reduce o se invierte? | — |
| 8 | Calidad | extremos y promociones marcados; prima repetida sin ellos | no borrar en silencio | borrar extremos |
| — | Alcance | precio relativo sostenido, no valor contable de la marca | solo hay precios al consumidor | inferir margen o ventas |


In [7]:
l_ = cob[(cob.comparación == P.COMPARA["lata"]["nombre"]) & (cob.canal == P.CANAL)].iloc[0]
f_ = cob[(cob.comparación == P.COMPARA["familiar"]["nombre"]) & (cob.canal == P.CANAL)].iloc[0]
display(Markdown(
    f"**Conclusión.** No hay Pepsi de 600 ml en el archivo; el par limpio es la lata de 355 ml, que las dos marcas tienen en "
    f"**{l_['tiendas con las dos']} supermercados** de {l_.municipios} municipios. La botella familiar (2.75 L contra 3 L) se compara por "
    f"litro en {f_['tiendas con las dos']} supermercados."))

**Conclusión.** No hay Pepsi de 600 ml en el archivo; el par limpio es la lata de 355 ml, que las dos marcas tienen en **184 supermercados** de 56 municipios. La botella familiar (2.75 L contra 3 L) se compara por litro en 144 supermercados.